# VAA 5차 : HAVAA(H5★) vs HAVAA + 보정 HAA(H4★)

- HAVAA = Chamberlain 'VAA Above Average' : 구종 안에서 투구 높이로 보정 (`vaa_aa` = vaa − (a + b·plate_z), 구종별 선형 잔차) — [FanGraphs VAA primer (2022-02-01)](https://blogs.fangraphs.com/a-visualized-primer-on-vertical-approach-angle-vaa/)
- 보정 HAA = Chamberlain 'HAA Above Average' : 구종 안에서 좌우 위치·투구손·좌우 릴리스 위치로 보정 (`haa_aa` = haa − (a + b·plate_x + c·release_pos_x), 구종 × 투구손별 선형 잔차) — [FanGraphs HAA primer (2023-11-22)](https://blogs.fangraphs.com/a-visual-primer-on-horizontal-approach-angle-haa/)
- 회귀식 형태는 원문 비공개 → 선형으로 구현. 모두 2025 학습기간 적합, F 와 같은 전체 튜닝(n_iter = 12)
- 누수를 '개별 투구 위치'(투수·구종 시즌 평균에서 벗어난 몫)와 '투수·구종 평소 위치'(시즌 평균)로 나눠 측정

In [ ]:
import os, gc, time, json, warnings, urllib.request, joblib
os.environ.setdefault('LOKY_MAX_CPU_COUNT', str(os.cpu_count()))
import numpy as np
import pandas as pd
import lightgbm as lgb
from scipy import stats
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import stuff_pipeline as sp
import export_web_data as ew

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)
SEED = sp.SEED
OUT = os.path.join('outputs', 'vaa_exp')
CACHE = os.path.join(OUT, 'cache')
T0 = time.time()
num = lambda s: s.to_numpy('float64', na_value=np.nan)

F_FEATS = [f for f in sp.STUFF_FEATURES if f not in ('vx0', 'vz0', 'ax', 'ay', 'az', 'vy0')]
WEB = ['pa_event', 'is_swing', 'is_whiff', 'result', 'xwoba', 'plate_z_norm']
COLS = sorted(set(F_FEATS) | {'pitcher', 'game_date', 'description', 'type', 'estimated_woba_using_speedangle', 'zone', 'plate_x', 'plate_z',
                              'plate_x_r', 'vaa', 'vaa_adj', 'haa', 'waste', 'raw_F'} | set(WEB))
fr = {}
for y in (2024, 2025, 2026):
    cols = COLS + (['y3', 'split'] if y == 2025 else [])
    fr[y] = pd.read_parquet(os.path.join(CACHE, f'frame_{y}.parquet'), columns=cols)
    fr[y]['raw_F'] = fr[y]['raw_F'].astype('float32')
valid_start = pd.Timestamp(pd.read_json(os.path.join(CACHE, 'meta.json'), typ='series')['valid_start'])
is_tr = fr[2025]['game_date'] < valid_start
print('로드', {y: (len(f), f.shape[1]) for y, f in fr.items()}, f'메모리 약 {sum(f.memory_usage(deep=True).sum() for f in fr.values()) / 1e9:.2f} GB')

In [ ]:
def fit_resid(df_fit, yname, xs, keys):
    coef = {}
    for k, g in df_fit.groupby(keys, observed=True):
        g = g[[yname] + xs].astype(float).dropna()
        if len(g) >= 500:
            A = np.column_stack([np.ones(len(g))] + [g[x].to_numpy() for x in xs])
            coef[tuple(map(str, k if isinstance(k, tuple) else (k,)))] = np.linalg.lstsq(A, g[yname].to_numpy(), rcond=None)[0]
    return coef

def apply_resid(d, yname, xs, keys, coef, out):
    res = np.full(len(d), np.nan)
    kv = pd.Series(list(zip(*[d[k].astype(str) for k in keys])), index=d.index)
    X = np.column_stack([np.ones(len(d))] + [num(d[x]) for x in xs])
    yv = num(d[yname])
    for k, b in coef.items():
        m = (kv == k).to_numpy()
        res[m] = yv[m] - X[m] @ b
    d[out] = res

for y in fr:   # HAA 보정은 투구손별로 따로 적합하므로 좌우 반전 전 원값 사용
    lefty = (fr[y]['p_throws'] == 'L').to_numpy()
    fr[y]['haa_raw'] = np.where(lefty, -num(fr[y]['haa']), num(fr[y]['haa']))
    fr[y]['rel_x_raw'] = np.where(lefty, -num(fr[y]['release_pos_x']), num(fr[y]['release_pos_x']))
c_aa = fit_resid(fr[2025][is_tr], 'vaa', ['plate_z'], ['pitch_type'])
c_ha = fit_resid(fr[2025][is_tr], 'haa_raw', ['plate_x', 'rel_x_raw'], ['pitch_type', 'p_throws'])
for y in fr:
    apply_resid(fr[y], 'vaa', ['plate_z'], ['pitch_type'], c_aa, 'vaa_aa')
    apply_resid(fr[y], 'haa_raw', ['plate_x', 'rel_x_raw'], ['pitch_type', 'p_throws'], c_ha, 'haa_aa')
d25 = fr[2025]
g = d25.groupby(['pitch_type', 'p_throws'], observed=True)
display(pd.DataFrame({'n': g.size(), 'haa_aa 표준편차': g['haa_aa'].std(),
                      'corr(haa_aa, plate_x)': g.apply(lambda q: q['haa_aa'].corr(q['plate_x'].astype(float))),
                      'corr(haa_aa, release_pos_x)': g.apply(lambda q: q['haa_aa'].corr(q['rel_x_raw']))}).query('n >= 2000').round(3))

# 누수 분리 : 개별 투구 위치(투수·구종 시즌 평균에서 벗어난 몫) / 평균 경향(투수·구종 시즌 평균)
for loc, nm in [('plate_x_r', 'px'), ('plate_z', 'pz')]:
    gm = d25.groupby(['pitcher', 'pitch_type'], observed=True)[loc].transform('mean')
    d25[f'{nm}_mean'] = gm.astype('float64')
    d25[f'{nm}_dev'] = (d25[loc] - gm).astype('float64')

In [ ]:
# 지표 함수 (3차와 같은 정의)
def reg_metrics(y, p):
    return {'RMSE': np.sqrt(mean_squared_error(y, p)), 'MAE': mean_absolute_error(y, p), 'R2': r2_score(y, p)}

def within_corr(d, col, loc, min_n=100):
    x = pd.DataFrame({'g1': d['pitcher'].to_numpy(), 'g2': d['pitch_type'].astype(str).to_numpy(), 'a': num(d[col]), 'b': num(d[loc])}).dropna()
    x = x.assign(ab=x['a'] * x['b'], aa=x['a'] ** 2, bb=x['b'] ** 2)
    s = x.groupby(['g1', 'g2']).agg(n=('a', 'size'), a=('a', 'sum'), b=('b', 'sum'), ab=('ab', 'sum'), aa=('aa', 'sum'), bb=('bb', 'sum'))
    s = s[s['n'] >= min_n]
    r = (s['ab'] - s['a'] * s['b'] / s['n']) / np.sqrt((s['aa'] - s['a'] ** 2 / s['n']) * (s['bb'] - s['b'] ** 2 / s['n']))
    ok = r.notna() & np.isfinite(r)
    return np.average(r[ok], weights=s['n'][ok])

def leak_r2(feats, target, n=200_000):
    tr = d25[is_tr & d25[target].notna()].sample(n, random_state=SEED)
    va = d25[~is_tr & d25[target].notna()]
    m = lgb.LGBMRegressor(n_estimators=300, learning_rate=0.08, num_leaves=63, random_state=SEED, verbose=-1).fit(tr[feats], tr[target])
    return r2_score(va[target], m.predict(va[feats]))

def validity(d, col):
    d = d.assign(swing=d['description'].isin(ew.SWING_DESC), whiff=d['description'].isin(sp.WHIFF_DESC), xc=d['estimated_woba_using_speedangle'].where(d['type'] == 'X'))
    pp = d.groupby(['pitcher', 'pitch_type'], observed=True).agg(n=('swing', 'size'), s=(col, 'mean'), sw=('swing', 'sum'), wh=('whiff', 'sum'), xc=('xc', 'mean')).query('n >= 100')
    pt = d.groupby('pitcher').agg(n=('swing', 'size'), s=(col, 'mean'), sw=('swing', 'sum'), wh=('whiff', 'sum')).query('n >= 300')
    return {'투수×구종 ρ(구위, whiff%)': stats.spearmanr(pp['s'], pp['wh'] / pp['sw']).statistic,
            '투수×구종 ρ(구위, xwOBAcon)': stats.spearmanr(pp['s'], pp['xc'], nan_policy='omit').statistic,
            '투수 ρ(구위, whiff%)': stats.spearmanr(pt['s'], pt['wh'] / pt['sw']).statistic}

def pt_means(d, col, min_n=100):
    g = d.assign(pitch_type=d['pitch_type'].astype(str)).groupby(['pitcher', 'pitch_type'])[col].agg(['mean', 'count'])
    return g[g['count'] >= min_n]['mean']

def pt_whiff(d, min_n=100):
    g = d.assign(pitch_type=d['pitch_type'].astype(str), sw=d['description'].isin(ew.SWING_DESC), wh=d['description'].isin(sp.WHIFF_DESC)) \
         .groupby(['pitcher', 'pitch_type']).agg(n=('sw', 'size'), sw=('sw', 'sum'), wh=('wh', 'sum'))
    g = g[g['n'] >= min_n]
    return g['wh'] / g['sw']

def buckets(d, col):
    bins = pd.cut(d[col], ew.STUFF_BINS, labels=ew.STUFF_BIN_LABELS, right=False)
    return pd.DataFrame({lab: ew.outcome_rates(d[bins == lab]) for lab in ew.STUFF_BIN_LABELS}).T

samp = d25[d25['y3'].notna()]
fit_rows, va_rows = samp[samp['split'] != 'va'], samp[samp['split'] == 'va']
tr_rows, es_rows = samp[samp['split'] == 'tr'], samp[samp['split'] == 'es']
d26 = fr[2026]
aux26 = pd.DataFrame({'abs_x': d26['plate_x'].abs(), 'ball': d26['description'].isin(sp.BALL_DESC),
                      'hbp': d26['description'] == 'hit_by_pitch', 'out_zone': d26['zone'] >= 11}, index=d26.index)
PTS = ['FF', 'SI', 'FC', 'SL', 'ST', 'CH', 'FS', 'CU', 'KC']
model_F = joblib.load(os.path.join(OUT, 'stage3_F.joblib'))
FEATS = {'F': F_FEATS, 'H5': F_FEATS + ['vaa_aa'], 'H4': F_FEATS + ['vaa_aa', 'haa_aa']}
results, bias = {}, {}

def evaluate(k, m, feats):
    # 점수 열을 만들고 지표 계산 후, 필요 없는 열은 지운다 (메모리)
    sc = sp.StuffScaler().fit(m.predict(d25.loc[is_tr & ~d25['waste'], feats])) if k != 'F' else joblib.load(os.path.join(OUT, 'stage4_F.joblib'))
    for y in fr:
        raw = sc.transform(m.predict(fr[y][feats])).astype('float32')
        fr[y]['_raw'] = raw
        fr[y]['_s'] = pd.Series(raw, index=fr[y].index).where(~fr[y]['waste'])
    r = {}
    for a, v in reg_metrics(va_rows['y3'], m.predict(va_rows[feats])).items():
        r[f'검증 {a} (공통 표본)'] = v
    for a, v in validity(d25[~is_tr], '_s').items():
        r[f'검증 {a}'] = v
    for year in (2024, 2026):
        b = buckets(fr[year], '_s')
        r[f'{year} BA 격차'] = b.loc['<40', 'ba'] - b.loc['60+', 'ba']
        r[f'{year} Whiff 격차'] = b.loc['60+', 'whiff'] - b.loc['<40', 'whiff']
    for a, b in [(2024, 2025), (2025, 2026)]:
        mm = pd.concat([pt_means(fr[a], '_s'), pt_means(fr[b], '_s')], axis=1, join='inner')
        r[f'안정성 r(구위 {a}, 구위 {b})'] = mm.corr().iloc[0, 1]
        w = pd.concat([pt_means(fr[a], '_s'), pt_whiff(fr[b])], axis=1, join='inner')
        r[f'예측력 ρ(구위 {a}, whiff% {b})'] = stats.spearmanr(w.iloc[:, 0], w.iloc[:, 1]).statistic
    r['누수 R²(plate_z)'] = leak_r2(feats, 'plate_z')
    r['누수 R²(plate_x 우투 기준)'] = leak_r2(feats, 'plate_x_r')
    for nm, lab in [('px', '좌우'), ('pz', '높이')]:
        r[f'누수 R² {lab} : 개별 투구 위치'] = leak_r2(feats, f'{nm}_dev')
        r[f'누수 R² {lab} : 투수·구종 평소 위치'] = leak_r2(feats, f'{nm}_mean')
    e26 = d26.assign(abs_x=aux26['abs_x'])
    r['투수×구종 내 corr(점수, 높이 정규화)'] = within_corr(e26, '_s', 'plate_z_norm')
    r['투수×구종 내 corr(점수, |plate_x|)'] = within_corr(e26, '_s', 'abs_x')
    top = d26['_raw'] >= d26['_raw'].quantile(0.99)
    r['사구 평균 점수 (raw)'] = d26.loc[aux26['hbp'], '_raw'].mean()
    r['상위 1% 중 존 밖 볼 비율 (raw)'] = (aux26.loc[top, 'ball'] & aux26.loc[top, 'out_zone']).mean()
    bias[k] = {pt: within_corr(d26[d26['pitch_type'] == pt], '_s', 'plate_z_norm') for pt in PTS}
    c = m.booster_.predict(va_rows[feats].sample(5000, random_state=SEED), pred_contrib=True)[:, :-1]
    names = [{'vaa_aa': 'vaa', 'haa_aa': 'haa'}.get(f, f) for f in feats]
    for gname, cols in ew.EXPLAIN_GROUPS:
        idx = [i for i, f in enumerate(names) if f in cols]
        r[f'|SHAP|×1000 {gname}'] = np.abs(c[:, idx].sum(axis=1)).mean() * 1000 if idx else np.nan
    for y in fr:
        fr[y].drop(columns=['_raw', '_s'], inplace=True)
    gc.collect()
    results[k] = pd.Series(r)

In [ ]:
t = time.time()
evaluate('F', model_F, F_FEATS)
evaluate('H5★', joblib.load(os.path.join(OUT, 'stage3_H5t.joblib')), FEATS['H5'])
print(f'F·H5★ 평가 {time.time() - t:.0f}초')
t = time.time()
m = sp.fit_stuff_model(tr_rows[FEATS['H4']], tr_rows['y3'], es_rows[FEATS['H4']], es_rows['y3'], n_iter=12, verbose=False)
joblib.dump(m, os.path.join(OUT, 'stage3_H4t.joblib'))
print(f'H4★ best holdout rmse {m.tuning_results_["rmse"].min():.5f} (F 0.16879, H5★ 0.16879) · {time.time() - t:.0f}초')
evaluate('H4★', m, FEATS['H4'])

In [ ]:
r4 = pd.read_csv(os.path.join(OUT, 'comparison_round4.csv'), index_col=0, encoding='utf-8-sig')
res = pd.DataFrame(results).rename(columns={'F': 'F', 'H5★': 'H5★ F+HAVAA', 'H4★': 'H4★ F+HAVAA+보정 HAA'})
both = [i for i in res.index if i in r4.index]
print('재현 검사 (최대 절대 차이) : F', float((res.loc[both, 'F'] - r4.loc[both, 'F (재계산)']).abs().max()),
      '/ H5★', float((res.loc[both, 'H5★ F+HAVAA'] - r4.loc[both, 'H5★ Chamberlain VAA (튜닝)']).abs().max()))
res = res.join(r4[['A v2 (웹)']], how='left')[['A v2 (웹)', 'F', 'H5★ F+HAVAA', 'H4★ F+HAVAA+보정 HAA']]
display(res.round(4))
res.to_csv(os.path.join(OUT, 'comparison_round5.csv'), encoding='utf-8-sig')
leak = res.loc[[i for i in res.index if i.startswith('누수')]]
leak.to_csv(os.path.join(OUT, 'leak_split_round5.csv'), encoding='utf-8-sig')
display(pd.DataFrame(bias).round(3))
print(f'전체 {time.time() - T0:.0f}초')

### 판정 (2026-10-10)
> Jupyter 커널 연결 문제로 같은 코드를 일반 스크립트로 실행 (144초). 출력 : `outputs/vaa_exp/comparison_round5.csv`, `leak_split_round5.csv`. F·H5★ 는 4차와 같음 (차이 1e-15)

- haa_aa : 구종 × 투구손 안에서 plate_x·release_pos_x 와 상관 ≈ 0, 표준편차 0.27 ~ 0.49°
- 누수 분리 (피처로 맞히는 R²)

| | F | H5★ | H4★ |
|---|---|---|---|
| 좌우 : 개별 투구 위치 | 0.068 | 0.070 | **0.267** |
| 좌우 : 투수·구종 평소 위치 | 0.781 | 0.781 | 0.781 |
| 높이 : 개별 투구 위치 | 0.067 | 0.088 | 0.092 |
| 높이 : 투수·구종 평소 위치 | 0.871 | 0.872 | 0.872 |

- 보정 HAA 는 '평소 던지는 쪽'이 아니라 **투구 하나하나의 좌우 위치** 정보를 4배로 늘린다 (0.07 → 0.27). 선형 보정 잔차라도 릴리스 위치·수평 무브먼트와 합치면 실제 좌우 위치를 역산할 수 있기 때문
- 평소 위치(평균 경향)는 모든 F 계열이 0.78 / 0.87 로 같다 → 구종·릴리스·무브먼트가 원래 담고 있는 정보로, 진입각과 무관
- 그래도 점수가 좌우 위치에 휘둘리는 정도는 같음 (0.024 / 0.023) : 모델이 HAA 를 많이 쓰지 않음
- 성능 : H4★ 가 H5★ 보다 약간 낮음 (ρ whiff% 0.715 vs 0.719, 다음 시즌 예측 0.767 / 0.772 vs 0.771 / 0.778)
- **결론 : 보정 HAA 는 넣지 않는다. 최선은 그대로 H5★ (F + HAVAA)**. 운영 반영은 사용자 지시 후